# NFL Elo Rating Model

Prospective NFL Elo ratings with margin-of-victory adjustment, home-field advantage, offseason regression, and expanding-window parameter tuning.

In [1]:
# pip install nflreadpy pandas pyarrow


## 1. Setup and NFL Data

In [1]:
import numpy as np
import pandas as pd
import nflreadpy as nfl
from sklearn.metrics import accuracy_score, log_loss

#Load games from nfl library
games = nfl.load_schedules(seasons=True).to_pandas()
print(games.columns)
print(games.shape)
print(games.head())



Index(['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday',
       'gametime', 'away_team', 'away_score', 'home_team', 'home_score',
       'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis',
       'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest',
       'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds',
       'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game',
       'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id',
       'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee',
       'stadium_id', 'stadium'],
      dtype='object')
(7548, 46)
           game_id  season game_type  week     gameday weekday gametime  \
0  1999_01_MIN_ATL    1999       REG     1  1999-09-12  Sunday     None   
1   1999_01_KC_CHI    1999       REG     1  1999-09-12  Sunday     None   
2  1999_01_PIT_CLE    1999       REG     1  1999-09-12  Sunday     None   
3   1999_01_OAK_GB    1999      

In [5]:
games['season'].unique()

array([1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009,
       2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020,
       2021, 2022, 2023, 2024, 2025, 2026])

## 2. Core Elo Rating Functions

In [7]:
import numpy as np
import pandas as pd
from IPython.display import display


def null_coalesce(x, y):
    return y if x is None else x

def compress_exp_grid(min_val, max_val, n=10, curve=2):
    grid = np.unique(np.round(
        min_val + (max_val - min_val) *
        (np.exp(curve * np.linspace(0, 1, n)) - 1) / (np.exp(curve) - 1)
    ))
    return grid

def calc_mov_scale(games):
    mean_log_margin = np.log(
        (games["home_score"] - games["away_score"]).abs() + 1
    ).mean()
    return mean_log_margin

def nfl_home_result(home_score, away_score, overtime):
    conditions = [
        (home_score > away_score) & (overtime == 0),
        (home_score < away_score) & (overtime == 0),
        (home_score > away_score) & (overtime == 1),
        (home_score < away_score) & (overtime == 1),
        home_score == away_score
    ]
    choices = [1.00, 0.00, 0.75, 0.25, 0.50]
    return np.select(conditions, choices, default=np.nan)

def elo_season(games, K, ratings=None, scale=400, start_rating=1000,
               use_mov=True, mov_scale=1, use_home_adv=True, home_adv=0):

    ratings = {} if ratings is None else ratings.copy()
    n = len(games)

    if use_mov and (pd.isna(mov_scale) or mov_scale <= 0):
        mov_scale = 1
    if not use_home_adv:
        home_adv = 0

    elo_home_pre = np.zeros(n)
    elo_away_pre = np.zeros(n)
    elo_home_post = np.zeros(n)
    elo_away_post = np.zeros(n)
    p_home = np.zeros(n)

    for i in range(n):
        g = games.iloc[i]

        hid = str(g["home_team"])
        aid = str(g["away_team"])

        Ra = ratings.get(hid, start_rating)
        Rb = ratings.get(aid, start_rating)

        elo_home_pre[i] = Ra
        elo_away_pre[i] = Rb

        p = 1 / (1 + 10 ** ((Rb - Ra - home_adv) / scale))
        p_home[i] = p

        y = nfl_home_result(
            home_score=g["home_score"],
            away_score=g["away_score"],
            overtime=g["overtime"]
        )

        if pd.isna(y):
            raise ValueError(f"Invalid NFL result coding for game_id: {g['game_id']}")

        point_diff = abs(g["home_score"] - g["away_score"])
        if g["home_score"] > g["away_score"]:
          winner_elo_diff = (Ra + home_adv) - Rb
        elif g["home_score"] < g["away_score"]:
          winner_elo_diff = Rb - (Ra + home_adv)
        else:
          winner_elo_diff = 0

        if not use_mov:
          mov_mult = 1
        elif g["home_score"] == g["away_score"]:
          mov_mult = 1
        else:
          mov_mult = np.log(point_diff + 1) * (2.2 / (0.001 * winner_elo_diff + 2.2)) / mov_scale

        delta = K * (y - p) * mov_mult

        ratings[hid] = Ra + delta
        ratings[aid] = Rb - delta

        elo_home_post[i] = ratings[hid]
        elo_away_post[i] = ratings[aid]

    games_out = games.copy()
    games_out["elo_home_pre"] = elo_home_pre
    games_out["elo_away_pre"] = elo_away_pre
    games_out["elo_home_post"] = elo_home_post
    games_out["elo_away_post"] = elo_away_post
    games_out["p_home"] = p_home
    games_out["p_away"] = 1 - p_home

    return {"games": games_out, "ratings": ratings}

### Single-Season Elo Sanity Check

In [8]:
# Sort it chronologically, calculate its MOV scale, and run
test_games = games[
    (games["season"] == 2024) &
    games["home_score"].notna() &
    games["away_score"].notna()
].copy()

test_games = test_games.sort_values(["gameday", "game_id"]).reset_index(drop=True)

mov_scale = calc_mov_scale(test_games)

test_elo = elo_season(
    games=test_games,
    K=20,
    ratings={},
    scale=400,
    start_rating=1000,
    use_mov=True,
    mov_scale=mov_scale,
    use_home_adv=True,
    home_adv=50
)

test_elo["games"][
    ["gameday", "home_team", "away_team",
     "elo_home_pre", "elo_away_pre",
     "p_home", "elo_home_post", "elo_away_post"]
].head(20)

display(test_elo["ratings"])

#Sanity Check - average of elo values should still be around starting mean of 1000
print("Average ELO Rank: ",
      sum(test_elo["ratings"].values())
      / len(test_elo["ratings"]))

{'KC': 1082.1410739289765,
 'BAL': 1085.0029964991543,
 'PHI': 1136.6408942963722,
 'GB': 1048.6897287643872,
 'BUF': 1091.4035283122469,
 'ARI': 990.4287605298258,
 'NO': 940.4030492884059,
 'CAR': 916.1109742111105,
 'CLE': 900.5990508042443,
 'DAL': 960.9313654303023,
 'SEA': 1017.0539019733426,
 'DEN': 1040.6456537309425,
 'IND': 967.2698647453412,
 'HOU': 1019.3156627426816,
 'MIA': 989.0127740341378,
 'JAX': 936.9866554763992,
 'DET': 1098.161997374354,
 'LA': 1023.4345096424472,
 'LAC': 1040.0924434757628,
 'LV': 918.7375564290048,
 'NYG': 909.1271341245094,
 'MIN': 1060.2498979434017,
 'CIN': 1024.128651789547,
 'NE': 934.2967345444166,
 'ATL': 979.5072187351919,
 'PIT': 1012.2974025493453,
 'CHI': 949.9058464496032,
 'TEN': 902.2733258894802,
 'TB': 1037.9384527823422,
 'WAS': 1059.0909147475497,
 'SF': 962.6381881671542,
 'NYJ': 965.4837905880195}

Average ELO Rank:  1000.0


## 3. Offseason Regression and Evaluation

In [9]:
def carry_over(ratings, carry=0.6):
    if len(ratings) == 0:
        return ratings

    mu = np.mean(list(ratings.values()))
    return {team: carry * rating + (1 - carry) * mu
            for team, rating in ratings.items()}

def logloss(p, y, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log1p(-p))

### Carryover Sanity Check

In [10]:
#Sanity Checks
test_carry = carry_over(test_elo["ratings"], carry=0.75)

print(np.mean(list(test_carry.values())))
print(max(test_elo["ratings"].values()))
print(max(test_carry.values()))

1000.0
1136.6408942963722
1102.4806707222792


## 4. Elo Parameter Evaluation

In [11]:
def score_elo_params(all_games, seasons_tune, K, home_adv,
                     use_mov=True, mov_scale=1, use_home_adv=True,
                     scale=400, start_rating=1000, carry=0.6):

  ratings = {}
  ll_vec = []

  if not use_home_adv:
      home_adv = 0

  for s in seasons_tune:
      g_s = (all_games[all_games["season"] == s]
              .sort_values(["gameday", "game_id"])
              .reset_index(drop=True))

      res = elo_season(games=g_s, ratings=ratings, K=K, scale=scale,
                       start_rating=start_rating, use_mov=use_mov,
                       mov_scale=mov_scale, use_home_adv=use_home_adv,
                       home_adv=home_adv)

      games_res = res["games"]
      non_ties = games_res["home_score"] != games_res["away_score"]

      p = games_res.loc[non_ties, "p_home"].to_numpy()
      y = (games_res.loc[non_ties, "home_score"] >
            games_res.loc[non_ties, "away_score"]).astype(float).to_numpy()

      ll_vec.extend(logloss(p, y))
      ratings = carry_over(res["ratings"], carry=carry)

  return np.nanmean(ll_vec)

### Initial Parameter Test

In [12]:
test_score = score_elo_params(
    all_games=games,
    seasons_tune=[2021, 2022, 2023, 2024],
    K=20,
    home_adv=50,
    use_mov=True,
    mov_scale=calc_mov_scale(games[games["season"].isin([2021, 2022, 2023, 2024])]),
    use_home_adv=True,
    scale=400,
    start_rating=1000,
    carry=0.75
)

print(test_score)

0.6503069250902233


$0.6503$ is already meaningfully better than a naive 50/50 baseline:
$−log(.5)=0.6931$

## 5. Hyperparameter Tuning

In [13]:
from itertools import product

def tune_elo_params(all_games, seasons_tune, K_grid, home_adv_grid,
                    use_mov=True, mov_scale=1, use_home_adv=True,
                    scale=400, start_rating=1000, carry=0.6,
                    use_carry_grid=True,
                    carryover_grid=(0.35, 0.50, 0.65, 0.80)):

    if not use_home_adv:
        home_adv_grid = [0]

    carry_grid = carryover_grid if use_carry_grid else [carry]
    results = []

    for K, home_adv, carry_val in product(K_grid, home_adv_grid, carry_grid):
        ll = score_elo_params(
            all_games=all_games, seasons_tune=seasons_tune,
            K=K, home_adv=home_adv,
            use_mov=use_mov, mov_scale=mov_scale,
            use_home_adv=use_home_adv, scale=scale,
            start_rating=start_rating, carry=carry_val
        )

        results.append({"K": K, "home_adv": home_adv, "carry": carry_val,
                        "logloss": ll})

    results = pd.DataFrame(results).sort_values("logloss").reset_index(drop=True)

    best = results.iloc[0]

    return {"K_best": best["K"], "home_adv_best": best["home_adv"],
            "carry_best": best["carry"], "logloss": best["logloss"],
            "results": results}

### Tuning Sanity Check

In [14]:
K_grid = [10, 15, 20, 25]
home_adv_grid = [25, 40, 55, 70]

mov_scale = calc_mov_scale(
    games[games["season"].isin([2021, 2022, 2023, 2024])]
)

test_tune = tune_elo_params(
    all_games=games,
    seasons_tune=[2021, 2022, 2023, 2024],
    K_grid=K_grid,
    home_adv_grid=home_adv_grid,
    mov_scale=mov_scale,
    carryover_grid=[0.50, 0.65, 0.75, 0.85]
)

print(test_tune["K_best"])
print(test_tune["home_adv_best"])
print(test_tune["carry_best"])
print(test_tune["logloss"])

test_tune["results"].head(10)

25.0
40.0
0.85
0.6463982500974149


,K,home_adv,carry,logloss
0,25,40,0.85,0.646398
1,25,25,0.85,0.646476
2,25,40,0.75,0.646507
3,25,25,0.75,0.646573
4,25,40,0.65,0.647088
5,25,25,0.65,0.647146
6,25,55,0.85,0.648168
7,25,55,0.75,0.648294
8,25,40,0.50,0.648652
9,25,25,0.50,0.648702


## 6. Prospective Expanding-Window Elo

In [16]:
def run_expanding_k_elo(all_games, first_train_year=2003, first_eval_year=2010,
                        K_grid=None, home_adv_grid=None, scale=400,
                        start_rating=1000, carry=0.6, use_carry_grid=True,
                        carryover_grid=(0.35, 0.50, 0.65, 0.80),
                        param_tune_window=None, use_mov=True, use_home_adv=True,
                        projection_season=None):

    if K_grid is None:
        K_grid = compress_exp_grid(1, 50, n=10, curve=3)
    if home_adv_grid is None:
        home_adv_grid = compress_exp_grid(-4, 40, n=10, curve=2)
    if not use_home_adv:
        home_adv_grid = [0]

    K_grid = np.atleast_1d(K_grid)
    home_adv_grid = np.atleast_1d(home_adv_grid)
    fixed_param_mode = len(K_grid) == 1 and len(home_adv_grid) == 1 and not use_carry_grid

    seasons = sorted(all_games["season"].unique())
    eval_years = [s for s in seasons if s >= first_eval_year]

    # Include the current schedule season even if it has no completed games yet.
    if projection_season is not None and projection_season not in eval_years:
        eval_years = sorted(set(eval_years + [projection_season]))

    all_out = []
    param_out = []

    for yr in eval_years:
        train_years = [s for s in seasons if first_train_year <= s < yr]
        if len(train_years) == 0:
            raise ValueError(f"No training seasons available before eval year: {yr}")

        param_tune_years = (
            train_years if param_tune_window is None
            else train_years[-param_tune_window:]
        )

        train_games = all_games[all_games["season"].isin(train_years)]
        mov_scale = calc_mov_scale(train_games) if use_mov else np.nan

        if fixed_param_mode:
            K_best = K_grid[0]
            home_adv_best = home_adv_grid[0] if use_home_adv else 0
            carry_best = carry
            best_logloss = np.nan
        else:
            param_fit = tune_elo_params(
                all_games=all_games, seasons_tune=param_tune_years,
                K_grid=K_grid, home_adv_grid=home_adv_grid,
                use_mov=use_mov, mov_scale=mov_scale,
                use_home_adv=use_home_adv, scale=scale,
                start_rating=start_rating, carry=carry,
                use_carry_grid=use_carry_grid,
                carryover_grid=carryover_grid
            )

            K_best = param_fit["K_best"]
            home_adv_best = param_fit["home_adv_best"] if use_home_adv else 0
            carry_best = param_fit["carry_best"] if use_carry_grid else carry
            best_logloss = param_fit["logloss"]

        ratings = {}

        for s in train_years:
            season_games = (all_games[all_games["season"] == s]
                            .sort_values(["gameday", "gametime", "game_id"])
                            .reset_index(drop=True))

            res = elo_season(
                games=season_games, ratings=ratings, K=K_best,
                scale=scale, start_rating=start_rating,
                use_mov=use_mov, mov_scale=mov_scale,
                use_home_adv=use_home_adv, home_adv=home_adv_best
            )

            ratings = carry_over(res["ratings"], carry=carry_best)

        eval_games = (all_games[all_games["season"] == yr]
                      .sort_values(["gameday", "gametime", "game_id"])
                      .reset_index(drop=True))

        res_eval = elo_season(
            games=eval_games, ratings=ratings, K=K_best,
            scale=scale, start_rating=start_rating,
            use_mov=use_mov, mov_scale=mov_scale,
            use_home_adv=use_home_adv, home_adv=home_adv_best
        )

        games_out = res_eval["games"].copy()
        games_out["K_used"] = K_best
        games_out["home_adv_used"] = home_adv_best
        games_out["carry_used"] = carry_best
        games_out["mov_scale_used"] = mov_scale
        games_out["use_mov"] = use_mov
        games_out["use_home_adv"] = use_home_adv
        games_out["train_start"] = min(train_years)
        games_out["train_end"] = max(train_years)
        games_out["param_tune_start"] = min(param_tune_years)
        games_out["param_tune_end"] = max(param_tune_years)
        games_out["param_tune_n"] = len(param_tune_years)
        games_out["param_tune_window"] = param_tune_window

        all_out.append(games_out)

        param_out.append({
            "eval_year": yr,
            "train_start": min(train_years),
            "train_end": max(train_years),
            "param_tune_start": min(param_tune_years),
            "param_tune_end": max(param_tune_years),
            "param_tune_n": len(param_tune_years),
            "param_tune_window": param_tune_window,
            "K_best": K_best,
            "home_adv_best": home_adv_best,
            "carry_best": carry_best,
            "mov_scale": mov_scale,
            "use_mov": use_mov,
            "use_home_adv": use_home_adv,
            "use_carry_grid": use_carry_grid,
            "logloss": best_logloss
        })

    return {
        "games": pd.concat(all_out, ignore_index=True),
        "K_by_year": pd.DataFrame(param_out),
        "ratings": res_eval["ratings"],
        "current_season": eval_years[-1]
    }

### Prospective Validation Test

In [50]:
# Exclude Preseason and Unplayed Games
elo_df = games[games["game_type"].isin(["REG", "WC", "DIV", "CON", "SB"]) &
               games["home_score"].notna() &
               games["away_score"].notna()].copy()

test_expanding = run_expanding_k_elo(
    all_games=elo_df,
    first_train_year=2005,
    first_eval_year=2010,
    K_grid=[15, 20, 25],
    home_adv_grid=[25, 40, 55],
    scale=400,
    start_rating=1000,
    use_carry_grid=True,
    carryover_grid=[0.3, 0.5, 0.75, 0.90],
    param_tune_window=5,
    use_mov=True,
    use_home_adv=True
)

#Inspect
display(test_expanding["K_by_year"])
display(test_expanding["games"][["season", "gameday", "home_team", "away_team",
                         "elo_home_pre", "elo_away_pre", "p_home", "K_used",
                         "home_adv_used", "carry_used", "mov_scale_used"]].head())

#Sanity Check
test_expanding["K_by_year"][["eval_year", "train_start", "train_end",
                             "param_tune_start", "param_tune_end"]]

,eval_year,train_start,train_end,param_tune_start,param_tune_end,param_tune_n,param_tune_window,K_best,home_adv_best,carry_best,mov_scale,use_mov,use_home_adv,use_carry_grid,logloss
0,2015,2005,2014,2010,2014,5,5,25.0,55.0,0.75,2.286770,True,True,True,0.630626
1,2016,2005,2015,2011,2015,5,5,25.0,55.0,0.75,2.280921,True,True,True,0.629981
2,2017,2005,2016,2012,2016,5,5,25.0,55.0,0.75,2.270998,True,True,True,0.630361
3,2018,2005,2017,2013,2017,5,5,25.0,55.0,0.75,2.272583,True,True,True,0.629627
4,2019,2005,2018,2014,2018,5,5,25.0,55.0,0.75,2.265796,True,True,True,0.635983
5,2020,2005,2019,2015,2019,5,5,25.0,40.0,0.75,2.266069,True,True,True,0.639958
6,2021,2005,2020,2016,2020,5,5,25.0,40.0,0.75,2.263232,True,True,True,0.636180
7,2022,2005,2021,2017,2021,5,5,25.0,25.0,0.75,2.264539,True,True,True,0.638681
8,2023,2005,2022,2018,2022,5,5,25.0,25.0,0.75,2.255347,True,True,True,0.641931
9,2024,2005,2023,2019,2023,5,5,25.0,25.0,0.75,2.252809,True,True,True,0.646376


,season,gameday,home_team,away_team,elo_home_pre,elo_away_pre,p_home,K_used,home_adv_used,carry_used,mov_scale_used
0,2015,2015-09-10,NE,PIT,1138.834670,1051.193674,0.694471,25.0,55.0,0.75,2.28677
1,2015,2015-09-13,JAX,CAR,860.563661,1022.167847,0.351224,25.0,55.0,0.75,2.28677
2,2015,2015-09-13,NYJ,CLE,939.768983,927.307162,0.595883,25.0,55.0,0.75,2.28677
3,2015,2015-09-13,CHI,GB,949.319189,1081.912216,0.390155,25.0,55.0,0.75,2.28677
4,2015,2015-09-13,BUF,IND,1001.725333,1061.879564,0.492583,25.0,55.0,0.75,2.28677


,eval_year,train_start,train_end,param_tune_start,param_tune_end
0,2015,2005,2014,2010,2014
1,2016,2005,2015,2011,2015
2,2017,2005,2016,2012,2016
3,2018,2005,2017,2013,2017
4,2019,2005,2018,2014,2018
5,2020,2005,2019,2015,2019
6,2021,2005,2020,2016,2020
7,2022,2005,2021,2017,2021
8,2023,2005,2022,2018,2022
9,2024,2005,2023,2019,2023


In [51]:
test_expanding['games'].shape


(3044, 64)

### Out-of-Sample Performance

In [52]:
#Calculate out-of-sample performance of 2023-2026
eval_games = test_expanding["games"].copy()
non_ties = eval_games["home_score"] != eval_games["away_score"]

eval_games["actual_home_win"] = (
    eval_games["home_score"] > eval_games["away_score"]).astype(int)

eval_games["pred_home_win"] = (eval_games["p_home"] >= 0.5).astype(int)

eval_games["logloss"] = np.nan
eval_games.loc[non_ties, "logloss"] = logloss(
    eval_games.loc[non_ties, "p_home"],
    eval_games.loc[non_ties, "actual_home_win"]
)

performance_by_year = (eval_games.loc[non_ties].groupby("season").agg(
    games=("game_id", "size"),
    accuracy=("pred_home_win",
              lambda x: (x == eval_games.loc[x.index, "actual_home_win"]).mean()),
    logloss=("logloss", "mean")).reset_index())

print(performance_by_year)

    season  games  accuracy   logloss
0     2015    267  0.677903  0.643835
1     2016    265  0.622642  0.631258
2     2017    267  0.674157  0.626347
3     2018    265  0.622642  0.634991
4     2019    266  0.642857  0.645139
5     2020    268  0.638060  0.628978
6     2021    284  0.619718  0.645556
7     2022    282  0.631206  0.649989
8     2023    285  0.614035  0.655743
9     2024    285  0.677193  0.616287
10    2025    284  0.640845  0.643088
11    2026     16  0.625000  0.643501


In [53]:
overall_accuracy = (eval_games.loc[non_ties, "pred_home_win"] ==
                    eval_games.loc[non_ties, "actual_home_win"]).mean()

overall_logloss = eval_games.loc[non_ties, "logloss"].mean()

print("Accuracy:", overall_accuracy)
print("Log Loss:", overall_logloss)

Accuracy: 0.6417270929466051
Log Loss: 0.6384192985164235


In [49]:
accuracy_by_year = (
    eval_games.loc[non_ties]
    .groupby("season")
    .apply(lambda x: (x["pred_home_win"] == x["actual_home_win"]).mean())
    .reset_index(name="accuracy")
)

accuracy_by_year

C:\Users\schne\AppData\Local\Temp\ipykernel_21528\4113808235.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x["pred_home_win"] == x["actual_home_win"]).mean())


,season,accuracy
0,2015,0.677903
1,2016,0.622642
2,2017,0.674157
3,2018,0.622642
4,2019,0.642857
5,2020,0.623134
6,2021,0.619718
7,2022,0.620567
8,2023,0.617544
9,2024,0.680702


## 7. Final Prospective Elo Model

In [54]:
nfl_elo = run_expanding_k_elo(all_games=elo_df, first_train_year=2005,
                    first_eval_year=2010,
                    K_grid = compress_exp_grid(10, 50, n=10, curve=2),
                    home_adv_grid = compress_exp_grid(0, 60, n=10, curve=2),
                    scale=400, start_rating=1000, use_carry_grid=True,
                    carryover_grid = [0.5, 0.65, 0.75, 0.85, 0.90],
                    param_tune_window=5, use_mov=True, use_home_adv=True)


### Performance Check

In [55]:
eval_games = nfl_elo["games"].query("home_score != away_score").copy()

eval_games["actual"] = (eval_games["home_score"] > eval_games["away_score"]).astype(int)
eval_games["pred"] = (eval_games["p_home"] >= 0.5).astype(int)
eval_games["correct"] = eval_games["pred"] == eval_games["actual"]
eval_games["ll"] = -(eval_games["actual"] * np.log(eval_games["p_home"]) + (1 - eval_games["actual"]) * np.log1p(-eval_games["p_home"]))

print("Accuracy:", accuracy_score(eval_games["actual"], eval_games["pred"]))
print("Log Loss:", log_loss(eval_games["actual"], eval_games["p_home"]))

Accuracy: 0.6440677966101694
Log Loss: 0.631581384199866


In [56]:
performance_by_year = eval_games.groupby("season").agg(
    games=("game_id", "size"), accuracy=("correct", "mean"),
    logloss=("ll", "mean")).reset_index()

performance_by_year

,season,games,accuracy,logloss
0,2010,267,0.606742,0.656580
1,2011,267,0.644195,0.615581
2,2012,266,0.631579,0.626380
3,2013,266,0.650376,0.613642
4,2014,266,0.672932,0.605484
5,2015,267,0.662921,0.644138
6,2016,265,0.645283,0.624219
7,2017,267,0.644195,0.620617
8,2018,265,0.630189,0.633847
9,2019,266,0.642857,0.641224


### Tuned Parameter Selection


In [57]:
params = nfl_elo["K_by_year"].copy()
params[["eval_year", "K_best", "home_adv_best", "carry_best", "mov_scale", "logloss"]]

,eval_year,K_best,home_adv_best,carry_best,mov_scale,logloss
0,2010,50.0,46.0,0.65,2.300139,0.623984
1,2011,41.0,46.0,0.65,2.291686,0.633151
2,2012,50.0,60.0,0.65,2.292068,0.624772
3,2013,50.0,60.0,0.50,2.289206,0.626113
4,2014,50.0,60.0,0.50,2.280759,0.627403
5,2015,50.0,60.0,0.65,2.286770,0.623161
6,2016,50.0,60.0,0.65,2.280921,0.623192
7,2017,50.0,60.0,0.65,2.270998,0.623152
8,2018,50.0,60.0,0.65,2.272583,0.622612
9,2019,50.0,60.0,0.50,2.265796,0.630373


In [58]:
print("K values:")
print(params["K_best"].value_counts().sort_index())

print("\nHome advantage values:")
print(params["home_adv_best"].value_counts().sort_index())

print("\nCarry values:")
print(params["carry_best"].value_counts().sort_index())
params[["K_best", "home_adv_best", "carry_best"]].describe()

K values:
K_best
41.0     3
50.0    14
Name: count, dtype: int64

Home advantage values:
home_adv_best
26.0    4
35.0    2
46.0    3
60.0    8
Name: count, dtype: int64

Carry values:
carry_best
0.50     5
0.65    11
0.75     1
Name: count, dtype: int64


,K_best,home_adv_best,carry_best
count,17.000000,17.000000,17.000000
mean,48.411765,46.588235,0.611765
std,3.536574,14.586204,0.078121
min,41.000000,26.000000,0.500000
25%,50.000000,35.000000,0.500000
50%,50.000000,46.000000,0.650000
75%,50.000000,60.000000,0.650000
max,50.000000,60.000000,0.750000


### Rerun Elo Tuning (Tightened Grids)

In [59]:
projection_season = int(
    games.loc[
        games["game_type"].isin(["REG", "WC", "DIV", "CON", "SB"]),
        "season"
    ].max()
)

nfl_elo = run_expanding_k_elo(
    all_games=elo_df,
    first_train_year=2005,
    first_eval_year=2010,
    K_grid=compress_exp_grid(35, 60, n=7, curve=1),
    home_adv_grid=compress_exp_grid(20, 40, n=7, curve=1),
    scale=400,
    start_rating=1000,
    use_carry_grid=True,
    carryover_grid=[0.5, 0.65, 0.7],
    param_tune_window=5,
    use_mov=True,
    use_home_adv=True,
    projection_season=projection_season
)


In [60]:
eval_games = nfl_elo["games"].query("home_score != away_score").copy()
eval_games["actual"] = (eval_games["home_score"] > eval_games["away_score"]).astype(int)
eval_games["pred"] = (eval_games["p_home"] >= 0.5).astype(int)
eval_games["correct"] = eval_games["pred"] == eval_games["actual"]
eval_games["ll"] = -(eval_games["actual"] * np.log(eval_games["p_home"]) + (1 - eval_games["actual"]) * np.log1p(-eval_games["p_home"]))

print("Accuracy:", accuracy_score(eval_games["actual"], eval_games["pred"]))
print("Log Loss:", log_loss(eval_games["actual"], eval_games["p_home"]))

performance_by_year = eval_games.groupby("season").agg(
    games=("game_id", "size"), accuracy=("correct", "mean"),
    logloss=("ll", "mean")).reset_index()

performance_by_year

Accuracy: 0.6438387540082455
Log Loss: 0.6313353562491582


,season,games,accuracy,logloss
0,2010,267,0.595506,0.658188
1,2011,267,0.644195,0.615482
2,2012,266,0.642857,0.624570
3,2013,266,0.654135,0.617063
4,2014,266,0.695489,0.606505
5,2015,267,0.655431,0.642794
6,2016,265,0.645283,0.624956
7,2017,267,0.651685,0.621216
8,2018,265,0.596226,0.636618
9,2019,266,0.639098,0.635240


In [61]:
params = nfl_elo["K_by_year"].copy()
params[["eval_year", "K_best", "home_adv_best", "carry_best", "mov_scale", "logloss"]]

,eval_year,K_best,home_adv_best,carry_best,mov_scale,logloss
0,2010,54.0,40.0,0.65,2.300139,0.624485
1,2011,44.0,40.0,0.65,2.291686,0.633212
2,2012,54.0,40.0,0.50,2.292068,0.625243
3,2013,54.0,40.0,0.50,2.289206,0.626579
4,2014,54.0,40.0,0.50,2.280759,0.628337
5,2015,54.0,40.0,0.50,2.286770,0.624068
6,2016,54.0,40.0,0.50,2.280921,0.623886
7,2017,54.0,40.0,0.50,2.270998,0.624232
8,2018,49.0,40.0,0.65,2.272583,0.623507
9,2019,49.0,40.0,0.50,2.265796,0.631044


## 8. Current Elo Ratings

Use the current ratings returned directly by the final expanding Elo fit. This keeps the rankings on the same 2018-forward training/replay path as the fitted model instead of separately replaying all available historical seasons with the current season's parameters.


In [62]:
current_season = nfl_elo["current_season"]
params = nfl_elo["K_by_year"].set_index("eval_year").loc[current_season]

# Use the final expanding fit's current Elo state directly.
ratings = nfl_elo["ratings"].copy()

current_teams = pd.unique(
    games.loc[
        games["season"] == current_season,
        ["home_team", "away_team"]
    ].values.ravel()
)

# Match the production script: a newly appearing team starts at 1000.
ratings = {team: ratings.get(team, 1000) for team in current_teams}

current_ratings = (
    pd.Series(ratings, name="elo")
    .rename_axis("team")
    .reset_index()
    .sort_values("elo", ascending=False)
    .reset_index(drop=True)
)
current_ratings.insert(0, "rank", range(1, len(current_ratings) + 1))

current_ratings


,rank,team,elo
0,1,SEA,1148.241223
1,2,BUF,1096.448948
2,3,SF,1083.064544
3,4,JAX,1076.891276
4,5,NE,1069.377967
5,6,MIN,1062.725463
6,7,HOU,1056.039532
7,8,CHI,1053.889641
8,9,BAL,1053.626407
9,10,PHI,1041.965526


## 9. Daily Rankings and Projections

Use current Elo ratings to identify games scheduled for a selected date and calculate each team's pregame Elo and win probability.

In [63]:
as_of_date = pd.Timestamp.now(tz="America/Chicago").date()
rank_map = current_ratings.set_index("team")["rank"]

future_projections = games[
    (games["season"] == current_season) &
    games["game_type"].isin(["REG", "WC", "DIV", "CON", "SB"]) &
    games["home_score"].isna() &
    games["away_score"].isna()
].copy()

future_projections["home_elo"] = future_projections["home_team"].map(ratings)
future_projections["away_elo"] = future_projections["away_team"].map(ratings)
future_projections["home_rank"] = future_projections["home_team"].map(rank_map)
future_projections["away_rank"] = future_projections["away_team"].map(rank_map)

future_projections["p_home"] = 1 / (1 + 10 ** (
    (future_projections["away_elo"] - future_projections["home_elo"] - params["home_adv_best"]) / 400
))
future_projections["p_away"] = 1 - future_projections["p_home"]
future_projections["as_of_date"] = as_of_date

future_projections = future_projections[
    ["as_of_date", "week", "gameday", "gametime", "away_team", "home_team",
     "away_rank", "home_rank", "away_elo", "home_elo", "p_away", "p_home"]
].sort_values(["gameday", "gametime"]).reset_index(drop=True)

daily_projections = future_projections[
    pd.to_datetime(future_projections["gameday"]).dt.date == as_of_date
].reset_index(drop=True)

current_week = future_projections.iloc[0]["week"]
weekly_projections = future_projections[
    future_projections["week"] == current_week
].reset_index(drop=True)

daily_rankings = current_ratings.assign(as_of_date=as_of_date)

display(future_projections)    # every remaining game using current Elo
display(daily_projections)     # today's games
display(weekly_projections)    # current/upcoming NFL week
display(daily_rankings)        # current league-wide rankings

,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home
0,2026-09-17,2,2026-09-17,20:15,DET,BUF,13,2,1027.971842,1096.448948,0.360629,0.639371
1,2026-09-17,2,2026-09-20,13:00,CAR,ATL,31,20,920.506191,970.413419,0.385625,0.614375
2,2026-09-17,2,2026-09-20,13:00,NO,BAL,24,9,958.382193,1053.626407,0.325915,0.674085
3,2026-09-17,2,2026-09-20,13:00,MIN,CHI,6,8,1062.725463,1053.889641,0.468146,0.531854
4,2026-09-17,2,2026-09-20,13:00,CIN,HOU,16,7,997.687811,1056.039532,0.374174,0.625826
...,...,...,...,...,...,...,...,...,...,...,...,...
251,2026-09-17,18,2027-01-10,13:00,CHI,MIN,8,6,1053.889641,1062.725463,0.442922,0.557078
252,2026-09-17,18,2027-01-10,13:00,MIA,NE,27,5,928.013684,1069.377967,0.270476,0.729524
253,2026-09-17,18,2027-01-10,13:00,TB,NO,23,24,958.640538,958.382193,0.455874,0.544126
254,2026-09-17,18,2027-01-10,13:00,PHI,NYG,10,19,1041.965526,974.285838,0.552591,0.447409


,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home
0,2026-09-17,2,2026-09-17,20:15,DET,BUF,13,2,1027.971842,1096.448948,0.360629,0.639371


,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home
0,2026-09-17,2,2026-09-17,20:15,DET,BUF,13,2,1027.971842,1096.448948,0.360629,0.639371
1,2026-09-17,2,2026-09-20,13:00,CAR,ATL,31,20,920.506191,970.413419,0.385625,0.614375
2,2026-09-17,2,2026-09-20,13:00,NO,BAL,24,9,958.382193,1053.626407,0.325915,0.674085
3,2026-09-17,2,2026-09-20,13:00,MIN,CHI,6,8,1062.725463,1053.889641,0.468146,0.531854
4,2026-09-17,2,2026-09-20,13:00,CIN,HOU,16,7,997.687811,1056.039532,0.374174,0.625826
5,2026-09-17,2,2026-09-20,13:00,PIT,NE,14,5,1023.054678,1069.377967,0.390524,0.609476
6,2026-09-17,2,2026-09-20,13:00,GB,NYJ,17,28,986.898753,927.672912,0.540531,0.459469
7,2026-09-17,2,2026-09-20,13:00,CLE,TB,29,23,923.827151,958.640538,0.406404,0.593596
8,2026-09-17,2,2026-09-20,13:00,PHI,TEN,10,32,1041.965526,866.691284,0.696463,0.303537
9,2026-09-17,2,2026-09-20,16:05,JAX,DEN,4,12,1076.891276,1036.312996,0.513781,0.486219


,rank,team,elo,as_of_date
0,1,SEA,1148.241223,2026-09-17
1,2,BUF,1096.448948,2026-09-17
2,3,SF,1083.064544,2026-09-17
3,4,JAX,1076.891276,2026-09-17
4,5,NE,1069.377967,2026-09-17
5,6,MIN,1062.725463,2026-09-17
6,7,HOU,1056.039532,2026-09-17
7,8,CHI,1053.889641,2026-09-17
8,9,BAL,1053.626407,2026-09-17
9,10,PHI,1041.965526,2026-09-17


In [65]:
### Save Data
current_ratings.to_csv("current_elo_rankings.csv", index=False)
daily_projections.to_csv("daily_projections.csv", index=False)
weekly_projections.to_csv("weekly_projections.csv", index=False)
future_projections.to_csv("future_projections.csv", index=False)
# historical_elo.to_csv("nfl_elo_history.csv", index=False)